# Building Image Generation Applications (Azure OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on Microsoft Foundry.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

On Microsoft Foundry, **`gpt-image-2`** is the latest and most capable image model and is the recommended default. `gpt-image-1.5` and `gpt-image-1-mini` are also generally available.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## Ihre erste Bildgenerierungsanwendung erstellen

Was braucht man also, um eine Bildgenerierungsanwendung zu erstellen? Sie benötigen die folgenden Bibliotheken:

- **python-dotenv**, es wird dringend empfohlen, diese Bibliothek zu verwenden, um Ihre Geheimnisse in einer *.env*-Datei vom Code getrennt aufzubewahren.
- **openai**, diese Bibliothek verwenden Sie, um mit der OpenAI API zu interagieren.
- **pillow**, um mit Bildern in Python zu arbeiten.

Falls noch nicht geschehen, folgen Sie den Anweisungen auf der [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) Seite, um eine Microsoft Foundry-Ressource und ein Modell zu erstellen. Wählen Sie **gpt-image-2** als Modell (das neueste Azure OpenAI Bildmodell; DALL·E ist veraltet).

1. Erstellen Sie eine Datei *.env* mit folgendem Inhalt:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Finden Sie diese Informationen im Microsoft Foundry-Portal für Ihre Ressource im Bereich "Deployments".



1. Sammeln Sie die oben genannten Bibliotheken in einer Datei namens *requirements.txt* wie folgt:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Erstellen Sie anschließend eine virtuelle Umgebung und installieren Sie die Bibliotheken:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Für Windows verwenden Sie die folgenden Befehle, um Ihre virtuelle Umgebung zu erstellen und zu aktivieren:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Fügen Sie den folgenden Code in eine Datei namens *app.py* ein:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # Azure OpenAI (Microsoft Foundry) Client konfigurieren.
    # Bildmodelle benötigen eine aktuelle API-Version – überprüfen Sie die Foundry-Dokumentation für die benötigte Version Ihres Modells.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Erstellen Sie ein Bild mithilfe der Bildgenerierungs-API
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Geben Sie hier Ihren Prompt-Text ein
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # z. B. "gpt-image-2"
        )
        # Legen Sie das Verzeichnis für das gespeicherte Bild fest
        image_dir = os.path.join(os.curdir, 'images')

        # Wenn das Verzeichnis nicht existiert, erstellen Sie es
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialisieren Sie den Bildpfad (beachten Sie, dass der Dateityp png sein sollte)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image-Modelle liefern das Bild als base64 (b64_json), nicht als URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Bild im Standardbildbetrachter anzeigen
        image = Image.open(image_path)
        image.show()

    # Ausnahmen abfangen
    except BadRequestError as err:
        print(err)

    ```

Erklären wir diesen Code:

- Zuerst importieren wir die benötigten Bibliotheken, darunter die OpenAI-Bibliothek, die dotenv-Bibliothek, das base64-Modul und die Pillow-Bibliothek.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Anschließend laden wir die Umgebungsvariablen aus der *.env*-Datei.

    ```python
    # importiere dotenv
    dotenv.load_dotenv()
    ```

- Danach konfigurieren wir den Azure OpenAI (Microsoft Foundry) Client.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Als Nächstes erzeugen wir das Bild:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Geben Sie hier Ihren Eingabetext ein
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` Modelle geben das Bild als **base64**-String in `data[0].b64_json` zurück. Wir dekodieren es in Bytes und schreiben es in eine Datei — es gibt keine URL zum Herunterladen.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Schließlich öffnen wir das Bild und verwenden den Standard-Bildbetrachter, um es anzuzeigen:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Weitere Details zur Bilderzeugung

Sehen wir uns die Parameter von `images.generate` an:

- **prompt** ist der Textprompt, der verwendet wird, um das Bild zu erzeugen. Hier lautet er "Hase auf Pferd, hält einen Lutscher, auf einer nebligen Wiese, auf der Narzissen wachsen".
- **size** ist die Größe des erzeugten Bildes (z. B. `1024x1024`, `1536x1024`, `1024x1536` oder `"auto"`).
- **n** ist die Anzahl der erzeugten Bilder. Hier erzeugen wir eins.
- **model** ist der Deployment-Name Ihres Bildmodell (zum Beispiel `gpt-image-2`).

> Bildmodelle verwenden keinen `temperature`-Parameter — das ist eine Steuerung der Textgenerierung. Für mehr Vielfalt rufen Sie die API erneut auf; um weniger Vielfalt zu erzielen, machen Sie Ihren Prompt spezifischer.

## Zusätzliche Funktionen der Bilderzeugung

Sie haben gesehen, wie man mit ein paar Zeilen Python ein Bild erzeugt. `gpt-image` Modelle können ein bestehendes Bild auch **bearbeiten**. Indem Sie ein Bild, eine optionale **Maske** (die den zu ändernden Bereich markiert) und einen Prompt bereitstellen, können Sie einen Teil eines Bildes verändern — zum Beispiel unserem Hasen einen Hut aufsetzen.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# Bearbeitungen werden ebenfalls als Base64 zurückgegeben
edited_image = base64.b64decode(response.data[0].b64_json)
```

Das Basisbild zeigt nur den Hasen; das Endbild hat den Hut auf dem Hasen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
